This is to find the exact directory of the code, this may take a while btw

In [1]:
import os

for Root, Dirs, Files in os.walk("/kaggle/input"):
    Level = Root.replace("/kaggle/input", "").count(os.sep)

    if Level < 2:
        print(Root)
        for File in Files[:10]:
            print("   ", File)

/kaggle/input
/kaggle/input/competitions


KeyboardInterrupt: 

In [10]:
!ls /kaggle/input/competitions/rsna-knee-abnormality-detection

sample_submission.csv  test_series	train.csv     train_series.csv
test.csv	       test_series.csv	train_series


# Load the CSVs

In [11]:
import pandas as pd

DataRoot = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

Train = pd.read_csv(f"{DataRoot}/train.csv")
TrainSeries = pd.read_csv(f"{DataRoot}/train_series.csv")

print("Train:", Train.shape)
print("Train Series:", TrainSeries.shape)

display(Train.head())
display(TrainSeries.head())

Train: (4407, 14)
Train Series: (24371, 5)


,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,StudyInstanceUID,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.12343110195036213483...,1,1,Sagittal
1,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.13821229744997220641...,1,1,Axial
2,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.23084836536722595275...,0,0,Coronal
3,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.40734206102458723096...,1,1,Coronal
4,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.75714899997203615784...,0,0,Sagittal


# Looking at class distributuion

In [14]:
Labels = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

print(Train[TargetColumns].sum().sort_values(ascending=False))

Effusion            35.0
Synovitis           27.0
Medial Meniscus     26.0
ACL                 24.0
Lateral Meniscus    23.0
PF OA               21.0
Contusion           19.0
Fracture            18.0
Medial OA           15.0
Baker's             12.0
Lateral OA          11.0
MCL                  9.0
dtype: float64


# Basic Analysis

In [15]:
print("Train shape:", Train.shape)
print("Train Series shape:", TrainSeries.shape)

print("\nLabel distribution:")
print(Train[Labels].sum().sort_values())

print("\nSeries:")
print(TrainSeries.head())

print("\nAnatomical planes:")
print(TrainSeries["Anatomical_Plane"].value_counts())

print("\nFluid sensitive:")
print(TrainSeries["Fluid_Sensitive"].value_counts())

Train shape: (4407, 14)
Train Series shape: (24371, 5)

Label distribution:
MCL                  9.0
Lateral OA          11.0
Baker's             12.0
Medial OA           15.0
Fracture            18.0
Contusion           19.0
PF OA               21.0
Lateral Meniscus    23.0
ACL                 24.0
Medial Meniscus     26.0
Synovitis           27.0
Effusion            35.0
dtype: float64

Series:
                                    StudyInstanceUID  \
0  1.2.826.0.1.3680043.8.498.10004873229099053869...   
1  1.2.826.0.1.3680043.8.498.10004873229099053869...   
2  1.2.826.0.1.3680043.8.498.10004873229099053869...   
3  1.2.826.0.1.3680043.8.498.10004873229099053869...   
4  1.2.826.0.1.3680043.8.498.10004873229099053869...   

                                   SeriesInstanceUID  Fluid_Sensitive  \
0  1.2.826.0.1.3680043.8.498.12343110195036213483...                1   
1  1.2.826.0.1.3680043.8.498.13821229744997220641...                1   
2  1.2.826.0.1.3680043.8.498.230848365367225

In [16]:
for Index in range(10):
    print("=" * 100)
    print(Train.iloc[Index]["Report"])

Técnica: RMN de la rodilla. Resultados: Rotura de menisco interno. Signo de necrosis avascular subcondral en el cóndilo femoral medial. Artrosis femorotibial medial. Derrame. . Impresión: Rotura de menisco interno. Signo de necrosis avascular subcondral en el cóndilo femoral medial. Artrosis femorotibial medial. Derrame.
[DATE]: * MR Knie Rechts 15ch AA Klinische Inlichtingen: [DATE]. Diagnostische vraagstellling: Meniscusscheur/mediaal? Scanprotocol (DRB) : sag intermediair gewogen seq zonder en met fs, ax/ cor pd gewogen seq fs, cor T1 gewogen seq Bevindingen:
Hallazgos:
No hay alteraciones en significativas de la médula ósea.
Ligamentos cruzados y colaterales dentro de límites normales.
Aumento de señal del cuerno posterior del menisco medial que no impresiona contactar con la
superficie articular, sin caracteres propios de rotura.
Menisco lateral de morfología y señal conservada, sin signos de rotura.
Úlceras condrales de espesor parcial con fenómenos condrales reparativos de la re

In [17]:
print(Train["Report"].isna().sum())
print(Train["Report"].str.len().describe())

0
count    4407.000000
mean     1097.905605
std       693.948925
min        52.000000
25%       587.500000
50%       977.000000
75%      1459.500000
max      4743.000000
Name: Report, dtype: float64


In [18]:
print(Train[["StudyInstanceUID", "Report"]].head())

                                    StudyInstanceUID  \
0  1.2.826.0.1.3680043.8.498.10004873229099053869...   
1  1.2.826.0.1.3680043.8.498.10004945927472656027...   
2  1.2.826.0.1.3680043.8.498.10009278692606631573...   
3  1.2.826.0.1.3680043.8.498.10009639203170750274...   
4  1.2.826.0.1.3680043.8.498.10013663742400736029...   

                                              Report  
0  Técnica: RMN de la rodilla. Resultados: Rotura...  
1  [DATE]: * MR Knie Rechts 15ch AA Klinische Inl...  
2  Hallazgos:\nNo hay alteraciones en significati...  
3   In the medial compartment, the meniscus is no...  
4  CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...  


# To find the keywords

In [19]:
import re
import pandas as pd

FindingPatterns = {
    "ACL": [
        r"\bACL\b",
        r"anterior cruciate",
        r"ligament.*cruciate.*ant"
    ],
    
    "MCL": [
        r"\bMCL\b",
        r"medial collateral",
        r"ligament.*collateral.*med"
    ],
    
    "Medial Meniscus": [
        r"medial meniscus",
        r"meniscus med",
        r"menisco medial"
    ],
    
    "Lateral Meniscus": [
        r"lateral meniscus",
        r"meniscus lat",
        r"menisco lateral"
    ],
    
    "Medial OA": [
        r"medial.*osteo",
        r"medial.*arthro",
        r"medial compartment.*degenerative"
    ],
    
    "Lateral OA": [
        r"lateral.*osteo",
        r"lateral.*arthro",
        r"lateral compartment.*degenerative"
    ],
    
    "PF OA": [
        r"patellofemoral.*osteo",
        r"patellofemoral.*arthro",
        r"patellar.*osteoarthritis"
    ],
    
    "Effusion": [
        r"effusion",
        r"joint fluid",
        r"joint.*fluid",
        r"derrame",
        r"épanchement"
    ],
    
    "Synovitis": [
        r"synovitis",
        r"synovial.*hypertrophy"
    ],
    
    "Baker's": [
        r"baker",
        r"popliteal cyst",
        r"popliteal.*cyst"
    ],
    
    "Contusion": [
        r"bone contusion",
        r"bone bruise",
        r"bone marrow edema",
        r"contusion"
    ],
    
    "Fracture": [
        r"fracture",
        r"fracture.*bone",
        r"fraktur",
        r"fraktura"
    ]
}

for Label, Patterns in FindingPatterns.items():
    Pattern = "|".join(Patterns)
    
    Matches = Train["Report"].str.contains(
        Pattern,
        case=False,
        regex=True,
        na=False
    ).sum()
    
    print(f"{Label:20s}: {Matches:4d}")

ACL                 : 1578
MCL                 : 1596
Medial Meniscus     : 1496
Lateral Meniscus    : 1550
Medial OA           :  352
Lateral OA          :  345
PF OA               :  154
Effusion            : 2153
Synovitis           :  299
Baker's             : 1478
Contusion           :  889
Fracture            :  718
